In [2]:
import pandas as pd 
from sklearn.model_selection import train_test_split , GridSearchCV 
from sklearn.preprocessing import StandardScaler,LabelEncoder , OneHotEncoder 
from sklearn.pipeline import Pipeline 
from scikeras.wrappers import KerasClassifier 
from scikeras.wrappers import KerasClassifier, KerasRegressor
import tensorflow as tf 
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import Dense 
from tensorflow.keras.callbacks import EarlyStopping 
import pickle

In [3]:
data=pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)


In [5]:
label_encoder_gender=LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

onehot_encoder_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded=onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

data=pd.concat([data.drop('Geography',axis=1) ,geo_encoded_df],axis=1)

X=data.drop('Exited',axis=1)
y=data['Exited']


In [6]:
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

# Save pickle file 
with open('gender_encode_hyper.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)
with open('geo_encode_hyper.pkl','wb')as file:
    pickle.dump(onehot_encoder_geo,file)
with open('scaler_hyper.pkl','wb') as file:
    pickle.dump(scaler,file)

In [15]:
## Define a function to create  the model and try different parameters(KerasClassifier)
def create_model(neurons=32,layers=1):
    model = Sequential()
    model.add(Dense(neurons,activation='relu',input_shape=(X_train.shape[1],)))
    for _ in range(layers-1):
        model.add(Dense(neurons,activation='relu'))

    model.add(Dense(1,activation='sigmoid'))
    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model 

In [16]:
## Create a Keras classifier 

from scikeras.wrappers import KerasClassifier

model = KerasClassifier(
    layers=1,
    neurons=32,
    model=create_model,
    verbose=0
)


In [17]:
## Define the grid search parameters 
param_grid = {
    'layers': [1, 2, 3],
    'neurons': [16, 32, 64],
    'batch_size': [16, 32],
    'epochs': [50, 100]
}


In [ ]:
# Perform grid search 
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=3,
    n_jobs=-1
)

grid_result = grid.fit(X_train, y_train)


C:\Users\Madhav Yadav\OneDrive\Desktop\desktop1\Datascience_bootcamp\Deep_Learning\ANN\Churn_prediction\ann_venv\Lib\site-packages\joblib\externals\loky\process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


In [ ]:

# ## Print the best parameters 
print(f"Best : {grid_result.best_score_} using {grid_result.best_params_}")